In [ ]:
import os

# Physical GPU index. The 5th GPU is index 4.
VISIBLE_GPU = "4"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = VISIBLE_GPU

print(f"CUDA_VISIBLE_DEVICES={os.environ['CUDA_VISIBLE_DEVICES']}")
print("Restart the kernel after changing VISIBLE_GPU if torch was already imported.")


# Residual Trace Visualization

This notebook visualizes Huginn recurrent traces saved by `scripts/collect_huginn_residual_traces.py`.

Notation used below:
- `f_t` is the hidden state after recurrent step `t`
- residual is defined as `f_t - f_{t-1}`
- the heatmap therefore shows `||f_t - f_{t-1}||_2` for each generated token position and step


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import torch
import transformers

assert transformers.__version__.startswith("4.47."), (
    f"This notebook expects transformers 4.47.x, got {transformers.__version__}. "
    "Switch to the 'huginn-venv' kernel."
)

TRACE_PATH = Path("/data/hansen/serv12/HRM-Deq/recurrent-pretraining/outputs/residual_traces/gsm8k_cot_long_fewshot_config.samples_0_steps32/trace.pt")
METADATA_PATH = TRACE_PATH.with_name("metadata.json")

# Choose which stream to analyze.
# `residual_stream` is the raw state trajectory.
# `residual_stream_ln` is the same trajectory after final layer norm.
STREAM_KEY = "residual_stream"

assert TRACE_PATH.exists(), TRACE_PATH
assert METADATA_PATH.exists(), METADATA_PATH


In [ ]:
data = torch.load(TRACE_PATH, map_location="cpu")
metadata = json.loads(METADATA_PATH.read_text())

stream = data[STREAM_KEY].float()  # [position, step_plus_initial, hidden_dim]
step_residual = stream[:, 1:, :] - stream[:, :-1, :]  # residual = f_t - f_{t-1}
step_residual_norm = torch.linalg.vector_norm(step_residual, dim=-1)  # [position, step]

avg_step_residual = step_residual_norm.mean(dim=0)
std_step_residual = step_residual_norm.std(dim=0)
token_labels = metadata["generated_token_labels"]
generated_text = metadata["generated_text"]

print(f"trace file: {TRACE_PATH}")
print(f"stream key: {STREAM_KEY}")
print(f"stream shape: {tuple(stream.shape)}")
print(f"step_residual_norm shape: {tuple(step_residual_norm.shape)}")
print()
print("question:")
print(metadata["question"])
print()
print("generated text:")
print(generated_text)


In [ ]:
def compact_labels(labels, max_chars=14):
    compact = []
    for label in labels:
        label = label.replace("\n", "\\n")
        if len(label) > max_chars:
            label = label[: max_chars - 3] + "..."
        compact.append(label)
    return compact


heatmap = step_residual_norm.T.numpy()  # [step, position]
display_labels = compact_labels(token_labels)

fig_width = max(10, len(display_labels) * 0.45)
fig, ax = plt.subplots(figsize=(fig_width, 6))
im = ax.imshow(heatmap, aspect="auto", origin="lower", cmap="magma")
ax.set_title("Per-token residual heatmap")
ax.set_xlabel("Generated token position")
ax.set_ylabel("Step t (residual = f_t - f_{t-1})")
ax.set_xticks(np.arange(len(display_labels)))
if len(display_labels) <= 40:
    ax.set_xticklabels(display_labels, rotation=90, fontsize=8)
else:
    ax.set_xticklabels([str(i) for i in range(len(display_labels))], rotation=90, fontsize=8)
ax.set_yticks(np.arange(heatmap.shape[0]))
ax.set_yticklabels(np.arange(1, heatmap.shape[0] + 1))
cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label("L2 norm of f_t - f_{t-1}")
fig.tight_layout()
plt.show()


In [ ]:
x = np.arange(1, avg_step_residual.numel() + 1)
mean_curve = avg_step_residual.numpy()
std_curve = std_step_residual.numpy()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(x, mean_curve, linewidth=2.5, color="#1f77b4", label="Mean across positions")
ax.fill_between(x, mean_curve - std_curve, mean_curve + std_curve, color="#1f77b4", alpha=0.2, label="±1 std")
ax.set_title("Average residual magnitude across positions")
ax.set_xlabel("Step t")
ax.set_ylabel("Average ||f_t - f_{t-1}||_2")
ax.set_xticks(x)
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
# Optional: inspect a few token trajectories individually.
mean_per_position = step_residual_norm.mean(dim=1)
topk = min(8, mean_per_position.numel())
focus_positions = torch.topk(mean_per_position, k=topk).indices.tolist()

fig, ax = plt.subplots(figsize=(9, 5))
for pos in focus_positions:
    label = f"pos={pos}: {display_labels[pos]!r}"
    ax.plot(x, step_residual_norm[pos].numpy(), linewidth=1.8, label=label)

ax.set_title("Selected token trajectories")
ax.set_xlabel("Step t")
ax.set_ylabel("||f_t - f_{t-1}||_2")
ax.grid(alpha=0.25)
ax.legend(fontsize=8, ncols=2)
fig.tight_layout()
plt.show()
